# Single-Class YOLO Training — Microplastic Detection (Colab T4)

> **Pipeline: YOLO (detect "microplastic") → EfficientNet (classify fiber/film/fragment)**

## Overview

This notebook trains a **single-class YOLOv8m** detector that learns to find all microplastic
particles regardless of type. Classification into fiber/film/fragment is handled downstream
by EfficientNet. This two-stage approach improves both detection recall and classification accuracy.

## Training Configuration

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| Model | YOLOv8m | Best accuracy/VRAM trade-off for T4 |
| Image size | 1280 | High resolution for small objects |
| Batch size | 2 | Maximum for YOLOv8m @ 1280 on T4 (15 GB) |
| Epochs | 200 | With early stopping (patience=50) |
| Optimizer | AdamW | Better generalisation on small datasets |
| Classes | 1 (microplastic) | Single-class detection |
| Cache | Disk | Saves RAM; uses local SSD for fast I/O |
| AMP | Enabled | ~2× throughput, halves activation memory |
| nbs | 64 | Gradient accumulation compensates small batch |

## Dataset

| Split | Images |
|-------|--------|
| Train | 800 (augmented) |
| Val | 200 |

## Google Drive Structure

```
MyDrive/mp-detect/
├── data/
│   └── yolo_augmented_single/
│       ├── dataset.yaml
│       ├── images/
│       │   ├── train/   (800 images)
│       │   └── val/     (200 images)
│       └── labels/
│           ├── train/
│           └── val/
└── experiments/
    └── yolo/            ← trained weights saved here
```

## 1. Environment Setup

In [ ]:
# ==============================================================================
# 1. ENVIRONMENT SETUP
# Mount Google Drive and install dependencies.
# ==============================================================================

from google.colab import drive
drive.mount('/content/drive')

!pip install ultralytics>=8.1.0 --quiet

import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")
print("Environment setup complete.")

## 2. GPU Verification & Reproducibility

**T4 Memory Budget (15 GB VRAM):**

| Configuration | Peak VRAM | Status |
|---------------|-----------|--------|
| YOLOv8m @ 1280, batch=2, AMP | ~11–13 GB | Safe |
| YOLOv8m @ 1280, batch=4, AMP | ~18+ GB | OOM |
| YOLOv8l @ 1280, batch=2, AMP | ~16+ GB | OOM |

**RAM Budget (~12.7 GB):**
- `cache='disk'` stores decoded images on local SSD instead of RAM
- Workers limited to 2 (Colab has 2 vCPUs)
- nbs=64 handles gradient accumulation so small batch still trains effectively

In [ ]:
# ==============================================================================
# 2. GPU VERIFICATION & REPRODUCIBILITY
# ==============================================================================

import torch
import random
import numpy as np
import os

assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime > Change runtime type > GPU."
)

gpu_name   = torch.cuda.get_device_name(0)
gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU           : {gpu_name}")
print(f"VRAM          : {gpu_mem_gb:.1f} GB")
print(f"PyTorch       : {torch.__version__}")
print(f"CUDA          : {torch.version.cuda}")

# Check available RAM
import psutil
ram_gb = psutil.virtual_memory().total / 1e9
ram_avail_gb = psutil.virtual_memory().available / 1e9
print(f"RAM total     : {ram_gb:.1f} GB")
print(f"RAM available : {ram_avail_gb:.1f} GB")

# Check local disk space (for disk cache)
disk = psutil.disk_usage('/content')
print(f"Disk total    : {disk.total / 1e9:.1f} GB")
print(f"Disk free     : {disk.free / 1e9:.1f} GB")

if "T4" in gpu_name:
    print("\n[INFO] Tesla T4 detected (15 GB VRAM).")
    print("[INFO] Using batch=2 at imgsz=1280 with disk cache.")

# Deterministic seed
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print(f"\nSeed: {SEED}")
print("GPU verification complete.")

## 3. Dataset: Copy to Local Disk

Copy dataset from Google Drive to the Colab VM's **local SSD** (`/content/dataset`).
Training from local disk is significantly faster than reading directly from Drive (FUSE overhead).

The disk cache (`cache='disk'`) will also be stored locally, keeping RAM free.

**Expected Drive structure:**
```
MyDrive/mp-detect/data/yolo_augmented_single/
├── dataset.yaml
├── images/
│   ├── train/   (800 images)
│   └── val/     (200 images)
└── labels/
    ├── train/
    └── val/
```

In [ ]:
# ==============================================================================
# 3. COPY DATASET TO LOCAL DISK
# Sequential copy with Drive-disconnect resilience.
# ==============================================================================

import os
import shutil
import time
import yaml
from pathlib import Path
from google.colab import drive as _colab_drive

# ---------------------------------------------------------------------------
# Paths — edit DRIVE_PROJECT_PATH if your Drive layout differs
# ---------------------------------------------------------------------------
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/mp-detect"
DRIVE_DATASET_PATH = f"{DRIVE_PROJECT_PATH}/data/yolo_augmented_single"
LOCAL_DATASET_PATH = "/content/dataset"

OUTPUT_PATH = f"{DRIVE_PROJECT_PATH}/experiments/yolo"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# ---------------------------------------------------------------------------
# Resilient file copy (handles Drive FUSE disconnects)
# ---------------------------------------------------------------------------
def _copy_drive_dataset(src_str, dst_str, max_retries=5):
    src = Path(src_str)
    dst = Path(dst_str)

    for attempt in range(3):
        try:
            all_items = sorted(src.rglob("*"))
            break
        except OSError as e:
            if e.errno == 107 and attempt < 2:
                print(f"  [WARN] Drive disconnected during scan, remounting...")
                _colab_drive.mount("/content/drive", force_remount=True)
            else:
                raise

    for item in all_items:
        if item.is_dir():
            (dst / item.relative_to(src)).mkdir(parents=True, exist_ok=True)

    files  = [f for f in all_items if f.is_file()]
    total  = len(files)
    copied = skipped = failed = 0

    print(f"  Copying {total} files sequentially...")
    for i, src_file in enumerate(files):
        dst_file = dst / src_file.relative_to(src)
        dst_file.parent.mkdir(parents=True, exist_ok=True)

        if dst_file.exists() and dst_file.stat().st_size > 0:
            skipped += 1
            continue

        for attempt in range(max_retries):
            try:
                shutil.copy2(str(src_file), str(dst_file))
                copied += 1
                break
            except OSError as e:
                if e.errno != 107 or attempt == max_retries - 1:
                    print(f"\n  [SKIP] {src_file.name}: {e}")
                    failed += 1
                    break
                delay = 2 ** attempt
                print(f"\n  [Errno 107] Drive disconnected — remounting (retry {attempt+1}/{max_retries}, wait {delay}s)...")
                time.sleep(delay)
                _colab_drive.mount("/content/drive", force_remount=True)

        if (i + 1) % 100 == 0 or (i + 1) == total:
            print(f"  Progress: {i+1}/{total}  "
                  f"({copied} copied, {skipped} skipped, {failed} failed)  ", end="\r")

    print(f"\n  Done: {copied} copied, {skipped} already existed, {failed} failed.")
    if failed > 0:
        raise RuntimeError(f"{failed} files could not be copied. Re-run this cell to retry.")


print("Copying dataset from Google Drive to local VM disk...")
_copy_drive_dataset(DRIVE_DATASET_PATH, LOCAL_DATASET_PATH)

DATASET_PATH = LOCAL_DATASET_PATH
YAML_PATH    = f"{DATASET_PATH}/dataset.yaml"

# ---------------------------------------------------------------------------
# Fix dataset.yaml path for Colab
# ---------------------------------------------------------------------------
assert os.path.exists(YAML_PATH), f"dataset.yaml not found at {YAML_PATH}"

with open(YAML_PATH) as f:
    ds_config = yaml.safe_load(f)

if ds_config.get("path") != DATASET_PATH:
    print(f"[FIX] Updating dataset.yaml 'path':")
    print(f"       Old: {ds_config.get('path')}")
    print(f"       New: {DATASET_PATH}")
    ds_config["path"] = DATASET_PATH
    with open(YAML_PATH, "w") as f:
        yaml.dump(ds_config, f, default_flow_style=False)

print("\ndataset.yaml contents:")
print(yaml.dump(ds_config, default_flow_style=False))

## 3.1 Dataset Verification

In [ ]:
# ==============================================================================
# 3.1 DATASET VERIFICATION
# Count images/labels, detect mismatches, show label distribution.
# ==============================================================================

from pathlib import Path
from collections import Counter
import numpy as np

IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

for split in ["train", "val"]:
    img_dir = Path(DATASET_PATH) / "images" / split
    lbl_dir = Path(DATASET_PATH) / "labels" / split

    assert img_dir.exists(), f"Missing directory: {img_dir}"
    assert lbl_dir.exists(), f"Missing directory: {lbl_dir}"

    images = {f.stem for f in img_dir.iterdir() if f.suffix.lower() in IMG_EXTS}
    labels = {f.stem for f in lbl_dir.iterdir() if f.suffix == ".txt"}

    missing_labels = images - labels
    orphan_labels  = labels - images

    # Count objects per label file (single class = all class 0)
    total_objects = 0
    bbox_widths, bbox_heights = [], []
    for lbl_file in lbl_dir.glob("*.txt"):
        for line in lbl_file.read_text().strip().splitlines():
            parts = line.split()
            if len(parts) >= 5:
                total_objects += 1
                bbox_widths.append(float(parts[3]))
                bbox_heights.append(float(parts[4]))

    print(f"[{split:5s}]  images: {len(images):4d}  |  labels: {len(labels):4d}  |  "
          f"objects: {total_objects:5d}  |  "
          f"missing labels: {len(missing_labels)}  |  orphan labels: {len(orphan_labels)}")

    if bbox_widths:
        w_arr = np.array(bbox_widths)
        h_arr = np.array(bbox_heights)
        IMGSZ_REF = 1280
        print(f"  Bbox size (px @ {IMGSZ_REF}):")
        print(f"    Median : {np.median(w_arr)*IMGSZ_REF:.0f} x {np.median(h_arr)*IMGSZ_REF:.0f}")
        print(f"    Mean   : {np.mean(w_arr)*IMGSZ_REF:.0f} x {np.mean(h_arr)*IMGSZ_REF:.0f}")
        small_count = np.sum((w_arr * IMGSZ_REF < 32) & (h_arr * IMGSZ_REF < 32))
        print(f"    Objects < 32x32 px : {small_count} / {len(w_arr)} "
              f"({100 * small_count / len(w_arr):.1f}%)")

    if missing_labels:
        print(f"  WARNING: {len(missing_labels)} images have no matching label file.")

print("\nDataset verification complete.")

## 4. Model Training

### Key Memory-Saving Strategies

1. **`cache='disk'`**: Decoded images stored on local SSD (`/content`) instead of RAM. 
   Colab free tier has ~12.7 GB RAM but ~100+ GB local disk — disk cache is the safe choice.
2. **`batch=2` + `nbs=64`**: Small batch to fit VRAM, but YOLO auto-accumulates gradients 
   as if batch=64 (accumulate = nbs/batch = 32 steps). This gives stable training.
3. **`amp=True`**: Mixed precision (FP16) halves memory for activations and weights.
4. **`workers=2`**: Matches Colab's 2 vCPUs. More workers waste RAM on unused prefetch buffers.
5. **`multi_scale=False`**: Dynamic resolution changes cause unpredictable VRAM spikes at 1280.
6. **`save_period=25`**: Checkpoints to Drive every 25 epochs for crash recovery.

### Single-Class Advantages

- Simpler task → converges faster with less data
- No inter-class confusion (fiber vs film distinction is hard for YOLO)
- Higher recall — model focuses purely on "is this a microplastic?"
- Classification handled by EfficientNet on cropped regions (much easier task)

In [ ]:
# ==============================================================================
# 4. MODEL TRAINING — Single-Class YOLOv8m
# Optimised for Colab T4 with disk caching to save RAM.
# ==============================================================================

from ultralytics import YOLO

# ---------------------------------------------------------------------------
# Core hyperparameters (T4-safe @ imgsz=1280)
# ---------------------------------------------------------------------------
MODEL          = "yolov8m.pt"      # 25.9M params — best for T4
IMGSZ          = 1280              # High res for small object detection
BATCH_SIZE     = 2                 # Max for YOLOv8m @ 1280 on T4
EPOCHS         = 200
PATIENCE       = 50               # Early stopping
EXPERIMENT     = "mp_yolov8m_single_class"

# ---------------------------------------------------------------------------
# Optimizer — AdamW with cosine LR
# ---------------------------------------------------------------------------
OPTIMIZER      = "AdamW"
LR0            = 0.0005            # Lower LR for fine-tuning pretrained model
LRF            = 0.01              # Final LR = LR0 * 0.01
WEIGHT_DECAY   = 5e-4
WARMUP_EPOCHS  = 5.0
WARMUP_MOM     = 0.8
WARMUP_BIAS_LR = 0.1

# ---------------------------------------------------------------------------
# Augmentation — aggressive for small dataset
# ---------------------------------------------------------------------------
MOSAIC         = 1.0               # Combine 4 images → more objects per step
COPY_PASTE     = 0.3               # Paste objects into new contexts
MIXUP          = 0.2               # Blend images for regularisation
HSV_H          = 0.015
HSV_S          = 0.7
HSV_V          = 0.4
DEGREES        = 15.0
TRANSLATE      = 0.2
SCALE          = 0.5
SHEAR          = 5.0
PERSPECTIVE    = 0.0005
FLIPUD         = 0.5
FLIPLR         = 0.5
ERASING        = 0.4              # Random erasing for regularisation
CLOSE_MOSAIC   = 20               # Disable mosaic for final 20 epochs
LABEL_SMOOTH   = 0.0              # No smoothing needed for single class

# ---------------------------------------------------------------------------
# Loss weights — elevated box weight for small object precision
# ---------------------------------------------------------------------------
BOX_LOSS       = 7.5
CLS_LOSS       = 0.5
DFL_LOSS       = 1.5

# ---------------------------------------------------------------------------
# Load model
# ---------------------------------------------------------------------------
model = YOLO(MODEL)

print(f"{'='*70}")
print(f"  Single-Class YOLO Training — Microplastic Detection")
print(f"{'='*70}")
print(f"  Model        : {MODEL}")
print(f"  Classes      : 1 (microplastic)")
print(f"  Image size   : {IMGSZ}")
print(f"  Batch size   : {BATCH_SIZE} (nbs=64 → accumulate={64//BATCH_SIZE} steps)")
print(f"  Epochs       : {EPOCHS} (early stopping patience={PATIENCE})")
print(f"  Optimizer    : {OPTIMIZER}, lr0={LR0}, cos_lr=True")
print(f"  Cache        : disk (saves RAM, uses local SSD)")
print(f"  AMP          : Enabled")
print(f"  Output       : {OUTPUT_PATH}/{EXPERIMENT}")
print(f"{'='*70}")

# ---------------------------------------------------------------------------
# TRAIN
# ---------------------------------------------------------------------------
results = model.train(
    # ----- Dataset -----
    data=YAML_PATH,

    # ----- Core training -----
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMGSZ,
    device=0,
    workers=2,                      # Colab has 2 vCPUs
    seed=SEED,
    deterministic=True,

    # ----- Optimizer & LR schedule -----
    optimizer=OPTIMIZER,
    lr0=LR0,
    lrf=LRF,
    momentum=0.937,
    weight_decay=WEIGHT_DECAY,
    warmup_epochs=WARMUP_EPOCHS,
    warmup_momentum=WARMUP_MOM,
    warmup_bias_lr=WARMUP_BIAS_LR,
    cos_lr=True,

    # ----- Early stopping -----
    patience=PATIENCE,

    # ----- Data augmentation -----
    augment=True,
    hsv_h=HSV_H,
    hsv_s=HSV_S,
    hsv_v=HSV_V,
    degrees=DEGREES,
    translate=TRANSLATE,
    scale=SCALE,
    shear=SHEAR,
    perspective=PERSPECTIVE,
    flipud=FLIPUD,
    fliplr=FLIPLR,
    mosaic=MOSAIC,
    mixup=MIXUP,
    copy_paste=COPY_PASTE,
    close_mosaic=CLOSE_MOSAIC,
    erasing=ERASING,

    # ----- Loss weights -----
    box=BOX_LOSS,
    cls=CLS_LOSS,
    dfl=DFL_LOSS,

    # ----- Regularisation -----
    label_smoothing=LABEL_SMOOTH,
    dropout=0.1,                    # Dropout for small dataset
    nbs=64,                         # Nominal batch size → accumulate = 64/2 = 32

    # ----- Mixed precision -----
    amp=True,

    # ----- Memory management (CRITICAL for Colab) -----
    cache='disk',                   # Cache to local SSD, NOT RAM
    rect=False,
    multi_scale=False,              # Avoid unpredictable VRAM spikes

    # ----- Saving (to Google Drive) -----
    project=OUTPUT_PATH,
    name=EXPERIMENT,
    exist_ok=True,
    save=True,
    save_period=25,                 # Checkpoint every 25 epochs to Drive

    # ----- Logging -----
    verbose=True,
    plots=True,
)

print(f"\n{'='*70}")
print("Training complete.")
print(f"Best weights : {OUTPUT_PATH}/{EXPERIMENT}/weights/best.pt")
print(f"Last weights : {OUTPUT_PATH}/{EXPERIMENT}/weights/last.pt")
print(f"{'='*70}")

## 5. Validation & Metrics

In [ ]:
# ==============================================================================
# 5. VALIDATION
# ==============================================================================

from ultralytics import YOLO

BEST_WEIGHTS = f"{OUTPUT_PATH}/{EXPERIMENT}/weights/best.pt"
model = YOLO(BEST_WEIGHTS)

metrics = model.val(
    data=YAML_PATH,
    imgsz=IMGSZ,
    batch=BATCH_SIZE,
    conf=0.001,
    iou=0.6,
    max_det=1000,
    plots=True,
    save_json=True,
)

print(f"\n{'='*70}")
print("  VALIDATION RESULTS — Single-Class Microplastic Detection")
print(f"{'='*70}")
print(f"  mAP@0.50        : {metrics.box.map50:.4f}")
print(f"  mAP@0.50:0.95   : {metrics.box.map:.4f}")
print(f"  Precision (mean) : {metrics.box.mp:.4f}")
print(f"  Recall (mean)    : {metrics.box.mr:.4f}")
print(f"{'='*70}")

# Per-class (single class)
print(f"\n  {'Class':15s}  {'AP@0.50':>8s}  {'AP@0.50:0.95':>12s}")
print(f"  {'-'*15}  {'-'*8}  {'-'*12}")
print(f"  {'microplastic':15s}  {metrics.box.map50:.4f}      {metrics.box.map:.4f}")

print(f"\nValidation plots saved to: {OUTPUT_PATH}/{EXPERIMENT}/")

## 5.1 Training Curves & Sample Predictions

In [ ]:
# ==============================================================================
# 5.1 TRAINING CURVES & SAMPLE PREDICTIONS
# ==============================================================================

import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import Image, display

results_dir = Path(OUTPUT_PATH) / EXPERIMENT

# Display training plots
plot_files = [
    "results.png",
    "confusion_matrix_normalized.png",
    "PR_curve.png",
    "F1_curve.png",
]

for pf in plot_files:
    plot_path = results_dir / pf
    if plot_path.exists():
        print(f"\n--- {pf} ---")
        display(Image(filename=str(plot_path), width=800))
    else:
        print(f"[SKIP] {pf} not found.")

# Sample predictions on validation images
val_images_dir = Path(DATASET_PATH) / "images" / "val"
sample_images  = sorted(val_images_dir.glob("*"))[:6]

if sample_images:
    model = YOLO(BEST_WEIGHTS)
    preds = model(
        [str(p) for p in sample_images],
        imgsz=IMGSZ,
        conf=0.25,
        iou=0.45,
        max_det=500,
    )

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    import cv2
    for ax, result in zip(axes.flatten(), preds):
        img = cv2.cvtColor(result.plot(), cv2.COLOR_BGR2RGB)
        ax.imshow(img)
        ax.set_title(f"{len(result.boxes)} detections", fontsize=11)
        ax.axis("off")

    plt.suptitle("Sample Validation Predictions (conf >= 0.25)", fontsize=14, y=1.01)
    plt.tight_layout()
    save_path = results_dir / "sample_predictions_single_class.png"
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved: {save_path}")

print("\nVisualisation complete.")

## 6. Export & Copy Weights to Drive

Copy the best weights to a convenient location on Drive for use in the local pipeline.

In [ ]:
# ==============================================================================
# 6. EXPORT & BACKUP
# ==============================================================================

from ultralytics import YOLO
from pathlib import Path
import shutil

model = YOLO(BEST_WEIGHTS)

# Export ONNX
onnx_path = model.export(format="onnx", imgsz=IMGSZ, simplify=True)
print(f"ONNX exported : {onnx_path}")

# Verify weights on Drive
weights_dir = Path(OUTPUT_PATH) / EXPERIMENT / "weights"
for w in sorted(weights_dir.glob("*.pt")):
    size_mb = w.stat().st_size / 1e6
    print(f"Saved on Drive : {w}  ({size_mb:.1f} MB)")

# Also copy best.pt to experiments/yolo/best.pt for easy local download
easy_path = Path(DRIVE_PROJECT_PATH) / "experiments" / "yolo" / "best.pt"
shutil.copy2(str(BEST_WEIGHTS), str(easy_path))
print(f"\nCopied best.pt to: {easy_path}")

print(f"\nTo use locally, download:")
print(f"  Google Drive > mp-detect > experiments > yolo > best.pt")
print(f"  Place it at: experiments/yolo/best.pt")
print(f"\nExport & backup complete.")

## Appendix: Resume Training After Disconnection

If the Colab session disconnects, uncomment and run the cell below to resume.

In [ ]:
# ==============================================================================
# RESUME TRAINING AFTER DISCONNECTION
# Uncomment all lines below and run if session was interrupted.
# ==============================================================================

# from google.colab import drive
# drive.mount('/content/drive')
# !pip install ultralytics>=8.1.0 --quiet

# from ultralytics import YOLO

# DRIVE_PROJECT_PATH = "/content/drive/MyDrive/mp-detect"
# OUTPUT_PATH        = f"{DRIVE_PROJECT_PATH}/experiments/yolo"
# EXPERIMENT         = "mp_yolov8m_single_class"
# LAST_WEIGHTS       = f"{OUTPUT_PATH}/{EXPERIMENT}/weights/last.pt"

# # Re-copy dataset to local disk (Colab VM was reset)
# import shutil, os
# DRIVE_DATASET_PATH = f"{DRIVE_PROJECT_PATH}/data/yolo_augmented_single"
# LOCAL_DATASET_PATH = "/content/dataset"
# if not os.path.exists(LOCAL_DATASET_PATH):
#     shutil.copytree(DRIVE_DATASET_PATH, LOCAL_DATASET_PATH)
#     # Fix yaml path
#     import yaml
#     yaml_path = f"{LOCAL_DATASET_PATH}/dataset.yaml"
#     with open(yaml_path) as f:
#         cfg = yaml.safe_load(f)
#     cfg['path'] = LOCAL_DATASET_PATH
#     with open(yaml_path, 'w') as f:
#         yaml.dump(cfg, f, default_flow_style=False)

# model = YOLO(LAST_WEIGHTS)
# results = model.train(resume=True)
# print("Resumed training complete.")